In [178]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
import numpy as np


采样一部分匹配异常的数据观测，主要根据匹配长度差值采样

In [179]:
matched_orders = pd.read_csv('matched_results_parallel.csv')
print(matched_orders.columns)

error_df = pd.read_csv("map_matching_error_analysis.csv")
print(error_df.columns)

original_df = pd.read_csv("filtered_orders.csv")
print(original_df.columns)

Index(['point_sequence', 'order_id', 'matched_latitude', 'matched_longitude',
       'total_order_distance_m'],
      dtype='object')
Index(['order_id', 'num_points', 'original_path_length_m',
       'matched_path_length_m', 'path_length_difference_m',
       'mean_pointwise_error_m', 'median_pointwise_error_m',
       'max_pointwise_error_m', 'frechet_distance'],
      dtype='object')
Index(['driver_id', 'order_id', 'gps_time', 'longitude', 'latitude',
       'time_diff'],
      dtype='object')


In [180]:
def filter_abnormal_taxi_orders(orders_df,matched_df,road_network_path):
    """
    过滤异常出租车轨迹订单数据。

    如果一个订单的任何一个GPS点15米半径内没有路网，则该订单被视为异常并被移除。

    参数:
        orders_df (pd.DataFrame): 包含出租车订单数据的DataFrame。
                                  必须包含 ['order_id', 'longitude', 'latitude'] 列。
        road_network_path (str): OSMNX导出的路网CSV文件的路径。

    返回:
        pd.DataFrame: 过滤后的出租车订单数据DataFrame。
    """
    # 1. 加载路网数据并转换为GeoDataFrame
    road_network_df = pd.read_csv(road_network_path)
    # 从WKT格式的'geometry'列创建几何图形
    # OSMnx 导出的 geometry 列通常是 Well-Known Text (WKT) 格式
    road_network_gdf = gpd.GeoDataFrame(
        road_network_df,
        geometry=gpd.GeoSeries.from_wkt(road_network_df['geometry']),
        crs="EPSG:4326"  # 假设原始坐标系为WGS84
    )

    # 2. 将订单数据转换为GeoDataFrame
    # 从经纬度创建几何图形点
    geometry = [Point(xy) for xy in zip(orders_df['longitude'], orders_df['latitude'])]
    orders_gdf = gpd.GeoDataFrame(
        orders_df,
        geometry=geometry,
        crs="EPSG:4326"  # 假设原始坐标系为WGS84
    )

    # 3. 转换坐标系以进行精确的米单位缓冲
    # 选择一个合适的UTM区域，这里以一个示例区域为例，您可能需要根据数据所在地区进行调整
    # 例如，对于中国大部分地区，可以选择一个通用的UTM区域或者根据省份选择
    # 这里我们估算一个中心点并获取其UTM区域
    utm_crs = orders_gdf.estimate_utm_crs()
    orders_gdf_proj = orders_gdf.to_crs(utm_crs)
    road_network_gdf_proj = road_network_gdf.to_crs(utm_crs)

    # 4. 为每个GPS点创建15米的缓冲区
    orders_gdf_proj['buffer'] = orders_gdf_proj.geometry.buffer(10)

    # 5. 使用空间连接（spatial join）检查缓冲区和路网的相交情况
    # 将缓冲区设置为主要的几何列进行连接
    orders_gdf_proj = orders_gdf_proj.set_geometry('buffer')

    # 'inner'连接会保留所有缓冲区与路网相交的点
    # 我们需要找到不相交的点，所以先进行'left'连接，然后检查未匹配上的
    join_result = gpd.sjoin(orders_gdf_proj, road_network_gdf_proj, how="left", predicate="intersects")

    # 6. 识别包含异常点的订单ID
    # 如果'index_right'列为NaN，说明该点没有与任何道路相交
    abnormal_points = join_result[join_result['index_right'].isna()]
    abnormal_order_ids = abnormal_points['order_id'].unique()
    print(abnormal_order_ids)

    print(f"发现 {len(abnormal_order_ids)} 个异常订单ID。")

    # 7. 从原始DataFrame中过滤掉所有异常订单
    filtered_orin_df = orders_df[~orders_df['order_id'].isin(abnormal_order_ids)]
    filtered_matched_df = matched_df[matched_df['order_id'].isin(abnormal_order_ids)]
    return filtered_orin_df, filtered_matched_df


In [181]:
from utils.filter_data_utils import haversine_np


def remove_duplicate_gps_points(df: pd.DataFrame, keep: str = 'first') -> pd.DataFrame:
    """
    移除每个订单中GPS位置（经纬度）完全相同的重复数据点。

    此函数会检查每个订单内部是否存在经纬度完全相同的GPS点。如果存在，
    它会根据 'keep' 参数的设置，保留第一个或最后一个出现的点，并移除其余的重复点。

    参数:
    - df (pd.DataFrame): 输入的DataFrame。
      必须包含列: ['order_id', 'longitude', 'latitude']。
    - keep (str): 当检测到重复项时，决定保留哪一个。
      - 'first': (默认) 保留第一次出现的记录。
      - 'last': 保留最后一次出现的记录。
      - False: 移除所有重复的记录。

    返回:
    - pd.DataFrame: 一个新的DataFrame，其中每个订单内位置重复的点已被移除。
    """
    # 检查必需的列是否存在
    required_columns = ['order_id', 'matched_longitude', 'matched_latitude']
    if not all(col in df.columns for col in required_columns):
        raise KeyError(f"输入DataFrame中缺少必需的列。需要 {required_columns}。")

    # 定义用于判断重复的列的子集。
    # 一条记录被认为是重复的，必须是它的 'order_id', 'longitude', 和 'latitude' 都与另一条记录相同。
    subset_cols = ['order_id', 'matched_longitude', 'matched_latitude']

    # 使用 drop_duplicates 方法。
    # 它会根据 subset_cols 中定义的列来查找重复行，并根据 keep 参数决定保留哪一行。
    # 因为我们将 'order_id' 包含在子集中，所以这个操作自然地只会在每个订单内部查找重复。
    # （即，不同订单中相同的经纬度点不会被认为是重复的）
    cleaned_df = df.drop_duplicates(subset=subset_cols, keep=keep)
    print(f"移除掉重复的订单数量有{len(df)-len(cleaned_df)} 条")

    return cleaned_df.copy()

def filter_orders_by_max_segment_distance(df: pd.DataFrame, max_segment_meters: int = 100) -> pd.DataFrame:
    """
    移除那些包含过长GPS段（跳点）的订单。

    此函数会计算每个订单中所有连续GPS点之间的距离。如果任何一个距离段
    大于指定的阈值，则该订单的全部数据都将被移除。

    参数:
    - df (pd.DataFrame): 输入的DataFrame。
      必须包含列: ['order_id', 'gps_time', 'longitude', 'latitude']。
    - max_segment_meters (int): 连续两点间允许的最大距离（单位：米）。
      默认为 100。

    返回:
    - pd.DataFrame: 一个新的DataFrame，其中不包含任何有跳点的订单。
    """
    # 检查必需的列是否存在
    required_columns = ['order_id', 'point_sequence', 'matched_longitude', 'matched_latitude']
    if not all(col in df.columns for col in required_columns):
        raise KeyError(f"输入DataFrame中缺少必需的列。需要 {required_columns}。")

    # 1. 创建副本并按订单和时间排序
    data = df.copy()
    data = data.sort_values(by=['order_id', 'point_sequence'])

    # 2. 获取每个点的“下一个点”的经纬度
    data['lon_next'] = data.groupby('order_id')['matched_longitude'].shift(-1)
    data['lat_next'] = data.groupby('order_id')['matched_latitude'].shift(-1)

    # 3. 计算每个GPS段的距离
    data['distance_segment'] = haversine_np(
        data['matched_longitude'],
        data['matched_latitude'],
        data['lon_next'],
        data['lat_next']
    )

    # 4. 找到每个订单中的【最大】分段距离
    max_distances = data.groupby('order_id')['distance_segment'].max()

    # 5. 找出所有分段距离都小于或等于阈值的订单
    #    单点订单的最大距离为NaN，不满足条件，会被自动移除。
    orders_to_keep = max_distances[max_distances <= max_segment_meters].index

    # 6. 从原始DataFrame中筛选出这些“好”订单的数据
    result_df = df[df['order_id'].isin(orders_to_keep)].copy()

    print(f"移除掉过长的订单数量有{len(df)-len(result_df)} 条")

    return result_df

In [182]:
def filter_trajectory_data(filtered_orders_path, matched_points_path, error_analysis_path):
    """
    根据路径长度差异筛选出租车订单轨迹数据。

    参数:
    filtered_orders_path (str): 原始出租车订单轨迹CSV文件的路径。
    matched_points_path (str): 匹配后的GPS轨迹CSV文件的路径。
    error_analysis_path (str): 地图匹配误差分析CSV文件的路径。

    返回:
    tuple: 包含两个DataFrame的元组 (filtered_original_df, filtered_matched_df)。
           - filtered_original_df: 筛选后的原始GPS轨迹数据。
           - filtered_matched_df: 筛选后的匹配GPS轨迹数据。
    """
    # 读取CSV文件到Pandas DataFrame
    try:
        original_orders_df = pd.read_csv(filtered_orders_path)
        matched_points_df = pd.read_csv(matched_points_path)
        error_analysis_df = pd.read_csv(error_analysis_path)
    except FileNotFoundError as e:
        print(f"错误: {e}")
        return None, None

    
    # 筛选出 path_length_difference_m 绝对值小于400米的订单
    filtered_error_df = error_analysis_df[error_analysis_df['path_length_difference_m'].abs() < 200]

    # 获取符合条件的订单ID列表
    valid_order_ids = filtered_error_df['order_id'].tolist()

    print(f"符合条件abs<200的订单数量有{len(valid_order_ids)}")

    # 根据订单ID筛选原始GPS轨迹和匹配后的GPS轨迹
    filtered_original_df = original_orders_df[original_orders_df['order_id'].isin(valid_order_ids)]
    filtered_matched_df = matched_points_df[matched_points_df['order_id'].isin(valid_order_ids)]
    
    
    
    # 移除重复点和相邻距离大于100m的订单
    filtered_matched_df = remove_duplicate_gps_points(filtered_matched_df, keep='first')
    # filtered_matched_df = filter_orders_by_max_segment_distance(filtered_matched_df, max_segment_meters=100)

    return filtered_original_df, filtered_matched_df


In [183]:
# 定义文件路径
filtered_orders_file = 'filtered_orders.csv'
matched_points_file = 'matched_results_parallel.csv'
error_analysis_file = 'map_matching_error_analysis.csv'

# 调用函数进行筛选
filtered_original_traces, filtered_matched_traces = filter_trajectory_data(
    filtered_orders_file,
    matched_points_file,
    error_analysis_file
)


# # 以下注释是随机采样订单，如果取消注释方便观察采样
# unique_orders = filtered_matched_traces['order_id'].unique()
# print(f"全部订单数量为{len(unique_orders)}")
# sampled_order_ids = np.random.choice(unique_orders, size=10, replace=False)
#
# print(sampled_order_ids)
#
# # 进行采样
# sampled_original_traces = filtered_original_traces[filtered_original_traces['order_id'].isin(sampled_order_ids)]
# sampled_matched_traces = filtered_matched_traces[filtered_matched_traces['order_id'].isin(sampled_order_ids)]
#
# print(f"sampled original dataframe len is {len(sampled_original_traces)}")
# print(f"sampled matched dataframe len is {len(sampled_matched_traces)}")
#
# sampled_original_traces.to_csv("sampled_original_traces.csv", index=False)
# sampled_matched_traces.to_csv("sampled_matched_traces.csv", index=False)

if len(filtered_original_traces['order_id'].unique()) == len(filtered_matched_traces['order_id'].unique()):
    print("筛选后的订单数量与匹配后的订单数量相同")
    # 保存筛选后的数据
    filtered_original_traces.to_csv("original_traces.csv", index=False)
    filtered_matched_traces.to_csv("matched_traces.csv", index=False)
else:
    print("筛选后的订单数量与匹配后的订单数量不相同")


符合条件abs<200的订单数量有3718
移除掉重复的订单数量有187 条
筛选后的订单数量与匹配后的订单数量相同
